In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries Loaded ✅")

Libraries Loaded ✅


In [3]:
transactions = pd.read_csv(r'C:\Users\HP\Desktop\Project 2 - Cohort - Retention - CLTV\Data\Row\transactions_raw.csv')
users = pd.read_csv(r'C:\Users\HP\Desktop\Project 2 - Cohort - Retention - CLTV\Data\Row\user_profiles.csv')

print("Transactions Shape:", transactions.shape)
print("Users Shape:", users.shape)
print("\nTransactions Columns:", list(transactions.columns))
print("Users Columns:", list(users.columns))

Transactions Shape: (3340, 9)
Users Shape: (2000, 8)

Transactions Columns: ['transaction_id', 'user_id', 'purchase_date', 'amount_inr', 'product', 'status', 'acquisition_channel', 'region', 'device']
Users Columns: ['user_id', 'signup_date', 'acquisition_channel', 'region', 'age_group', 'gender', 'plan_type', 'is_active']


In [5]:
print("=== TRANSACTIONS ===")
print(transactions.head())
print("\n=== USERS ===")
print(users.head())

=== TRANSACTIONS ===
  transaction_id    user_id purchase_date  amount_inr            product  \
0     TXN_207473  USR_00001    2023-11-24     1833.82           Pro Plan   
1     TXN_329258  USR_00001    2024-01-17      448.38  One-Time Purchase   
2     TXN_948749  USR_00002    2023-10-15     2172.81           Pro Plan   
3     TXN_201414  USR_00003    2023-06-24     5510.05    Enterprise Plan   
4     TXN_969693  USR_00004    2023-01-23      669.36  One-Time Purchase   

      status acquisition_channel   region   device  
0     failed       Google_Search    North   tablet  
1  completed       Google_Search    North   mobile  
2  completed       Meta_Facebook  Central   mobile  
3  completed               Email    South  desktop  
4  completed             Organic  Central   tablet  

=== USERS ===
     user_id signup_date acquisition_channel   region age_group  gender  \
0  USR_00001  2023-11-24       Google_Search    North     35-44   Other   
1  USR_00002  2023-10-15       Meta_Fac

In [7]:
# Cell 4 - Null Values Check
print("=== NULL VALUES ===")
print("\nTransactions:")
print(transactions.isnull().sum())
print("\nUsers:")
print(users.isnull().sum())

=== NULL VALUES ===

Transactions:
transaction_id         0
user_id                0
purchase_date          0
amount_inr             0
product                0
status                 0
acquisition_channel    0
region                 0
device                 0
dtype: int64

Users:
user_id                0
signup_date            0
acquisition_channel    0
region                 0
age_group              0
gender                 0
plan_type              0
is_active              0
dtype: int64


In [9]:
# Cell 5 - Filter Refunded/Failed (Issue #2)
print("Before filter:", transactions.shape)
print("Status counts:\n", transactions['status'].value_counts())

transactions_clean = transactions[transactions['status'] == 'completed'].copy()

print("\nAfter filter:", transactions_clean.shape)
print("Removed:", len(transactions) - len(transactions_clean), "rows")

Before filter: (3340, 9)
Status counts:
 status
completed    2211
failed        571
refunded      558
Name: count, dtype: int64

After filter: (2211, 9)
Removed: 1129 rows


In [11]:
# Cell 6 - Missing User IDs (Issue #3)
print("Missing user_ids:", transactions_clean['user_id'].isnull().sum())

transactions_clean = transactions_clean.dropna(subset=['user_id'])

print("After dropna:", transactions_clean.shape)
print("Missing user_ids now:", transactions_clean['user_id'].isnull().sum())

Missing user_ids: 0
After dropna: (2211, 9)
Missing user_ids now: 0


In [13]:
transactions_clean['purchase_date'] = pd.to_datetime(transactions_clean['purchase_date'])
users['signup_date'] = pd.to_datetime(users['signup_date'])

print("Date types fixed ✅")
print(transactions_clean['purchase_date'].dtype)

Date types fixed ✅
datetime64[ns]


In [15]:
first_purchase = transactions_clean.groupby('user_id')['purchase_date'].min().reset_index()
first_purchase.columns = ['user_id', 'first_purchase_date']
first_purchase['cohort_month'] = first_purchase['first_purchase_date'].dt.to_period('M')

print("=== COHORT MONTH SAMPLE ===")
print(first_purchase.head(10))
print("\nTotal unique cohorts:", first_purchase['cohort_month'].nunique())

=== COHORT MONTH SAMPLE ===
     user_id first_purchase_date cohort_month
0  USR_00001          2024-01-17      2024-01
1  USR_00002          2023-10-15      2023-10
2  USR_00003          2023-06-24      2023-06
3  USR_00004          2023-01-23      2023-01
4  USR_00005          2023-07-05      2023-07
5  USR_00006          2023-07-01      2023-07
6  USR_00008          2023-09-17      2023-09
7  USR_00010          2023-01-06      2023-01
8  USR_00012          2023-06-15      2023-06
9  USR_00013          2023-02-10      2023-02

Total unique cohorts: 16


In [17]:
transactions_clean = transactions_clean.merge(
    first_purchase[['user_id', 'cohort_month']],
    on='user_id',
    how='left'
)

print("Cohort month added ✅")
print(transactions_clean[['user_id', 'purchase_date', 'cohort_month']].head())

Cohort month added ✅
     user_id purchase_date cohort_month
0  USR_00001    2024-01-17      2024-01
1  USR_00002    2023-10-15      2023-10
2  USR_00003    2023-06-24      2023-06
3  USR_00004    2023-01-23      2023-01
4  USR_00005    2023-07-05      2023-07
